# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Awais-Ahmad10/flyrank_ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item, for one client, on one specific day (from fact_content_daily_performance).
I'm working with a mid-panel month, 2026-03, so I'm not testing on the sealed final month.

In [31]:
import duckdb
from google.colab import userdata

# Get your secret token safely (never typed directly here)
hf_token = userdata.get('HF_TOKEN')

# Connect DuckDB to Hugging Face
con = duckdb.connect()
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")

print("Connected! Ready to query.")

Connected! Ready to query.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: total_impressions, avg_position, total_sessions, total_engaged_sessions —
these are observed signals I can use to predict declining pages.

Label/proxy: is_declining, built from CTR and impression volume (a proxy, not a true future outcome).

Context: content_hash_id, client_hash_id — used only for joining/grouping, not as model inputs.

Excluded: total_clicks — excluded from the final honest model because it's part of the formula
used to build the CTR label, so including it would leak the answer into the features.

In [32]:
# Quick test: load the small dim_clients table (only 104 rows, so it's fast)
query = """
SELECT *
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
LIMIT 5
"""

df_test = con.sql(query).df()
df_test

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [33]:
query_grain = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) as row_count
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
"""

df_grain_check = con.sql(query_grain).df()
df_grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [34]:
query_scope = """
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT content_hash_id) as unique_pages,
    COUNT(DISTINCT client_hash_id) as unique_clients,
    MIN(report_date) as earliest_date,
    MAX(report_date) as latest_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""

df_scope = con.sql(query_scope).df()
df_scope

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_pages,unique_clients,earliest_date,latest_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [35]:
query_columns = """
SELECT *
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 1
"""
con.sql(query_columns).df().columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

In [21]:
query_availability = """
SELECT
    COUNT(*) as total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) as rows_with_gsc_data,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as rows_with_ga4_data
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""

df_availability = con.sql(query_availability).df()
df_availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_gsc_data,rows_with_ga4_data
0,9841378,3611061,413966


1. total_impressions — knowable at decision time because it's the sum of past search impressions already recorded before the decision.
2. total_clicks — knowable at decision time because it's the sum of past search clicks already recorded before the decision.
3. avg_position — knowable at decision time because it's the average search ranking position already observed in the feature window.
4. total_sessions — knowable at decision time because it's the sum of past GA4 sessions, though it will be NaN for clients without GA4 tracking (not a leak — just missing data).
5. total_engaged_sessions — knowable at decision time because it's the sum of past engaged sessions, same GA4 availability caveat as above.


In [22]:
query_features = """
SELECT
    content_hash_id,
    client_hash_id,
    SUM(gsc_impressions) as total_impressions,
    SUM(gsc_clicks) as total_clicks,
    AVG(gsc_avg_position) as avg_position,
    SUM(ga4_sessions) as total_sessions,
    SUM(ga4_engaged_sessions) as total_engaged_sessions
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY content_hash_id, client_hash_id
LIMIT 20
"""

df_features = con.sql(query_features).df()
df_features

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,total_impressions,total_clicks,avg_position,total_sessions,total_engaged_sessions
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,181.0,0.0,5.147402,NaN,NaN
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,34.0,0.0,4.909314,NaN,NaN
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,77.0,0.0,4.074107,NaN,NaN
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,329.0,0.0,5.177774,NaN,NaN
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,602.0,4.0,4.428747,NaN,NaN
5,content_614baf2af4330bd7,client_62f4a7e64f5e0096,772.0,1.0,4.685335,NaN,NaN
6,content_ceaec531566ffcfc,client_62f4a7e64f5e0096,82.0,0.0,8.978086,NaN,NaN
7,content_755d951187fcd70a,client_62f4a7e64f5e0096,1858.0,6.0,1.854929,NaN,NaN
8,content_7a7d3c7aa7cdfc5c,client_62f4a7e64f5e0096,16.0,0.0,15.000000,NaN,NaN
9,content_225dc9235023be5f,client_62f4a7e64f5e0096,488.0,1.0,17.148172,NaN,NaN


In [23]:
import numpy as np

query_leak_setup = """
SELECT
    content_hash_id,
    client_hash_id,
    SUM(gsc_impressions) as total_impressions,
    SUM(gsc_clicks) as total_clicks,
    AVG(gsc_avg_position) as avg_position,
    SUM(ga4_sessions) as total_sessions,
    SUM(ga4_engaged_sessions) as total_engaged_sessions
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY content_hash_id, client_hash_id
HAVING SUM(gsc_impressions) > 0
"""

df_leak = con.sql(query_leak_setup).df()

df_leak['ctr'] = df_leak['total_clicks'] / df_leak['total_impressions']
df_leak['is_declining'] = (df_leak['ctr'] < df_leak['ctr'].median()).astype(int)

print("Shape:", df_leak.shape)
print("Declining rate:", df_leak['is_declining'].mean())
df_leak.head()

Shape: (176738, 9)
Declining rate: 0.0


,content_hash_id,client_hash_id,total_impressions,total_clicks,avg_position,total_sessions,total_engaged_sessions,ctr,is_declining
0,content_05597932fe4da067,client_73cda7b4e4f265ea,57.0,0.0,2.714744,0.0,0.0,0.000000,0
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,7.209549,1.0,0.0,0.001073,0
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,149.0,0.0,6.481453,4.0,0.0,0.000000,0
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,453.0,0.0,2.987198,0.0,0.0,0.000000,0
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,6.724039,3.0,0.0,0.001066,0


In [24]:
print("Median CTR:", df_leak['ctr'].median())
print(df_leak['ctr'].describe())


Median CTR: 0.0
count    176738.000000
mean          0.004594
std           0.037760
min           0.000000
25%           0.000000
50%           0.000000
75%           0.002158
max           1.000000
Name: ctr, dtype: float64


In [25]:
# Better label: "low performing" = has real impressions but very poor CTR
# We require a minimum impression volume so we're not calling a page "declining"
# just because it had 1 impression and 0 clicks (that's noise, not decline)

df_leak['is_declining'] = (
    (df_leak['total_impressions'] >= 100) &   # enough exposure to judge fairly
    (df_leak['ctr'] < 0.01)                    # under 1% CTR = weak performance
).astype(int)

print("Declining rate:", df_leak['is_declining'].mean())
df_leak['is_declining'].value_counts()

Declining rate: 0.5477825934434021


,count
is_declining,
1,96814
0,79924


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Clean features: fill missing GA4 data with 0 (treat "no GA4 tracking" as 0 for now)
honest_features = ['total_impressions', 'total_clicks', 'avg_position', 'total_sessions', 'total_engaged_sessions']

df_model = df_leak.copy()
df_model[honest_features] = df_model[honest_features].fillna(0)

X = df_model[honest_features]
y = df_model['is_declining']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

honest_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("Honest AUC score:", honest_score)

Honest AUC score: 0.9924092290228306


In [27]:
# Remove total_clicks — it's part of the formula that created our label (ctr = clicks/impressions)
# Keeping it in would mean the model can nearly reconstruct the label directly
honest_features_v2 = ['total_impressions', 'avg_position', 'total_sessions', 'total_engaged_sessions']

X2 = df_model[honest_features_v2]
y2 = df_model['is_declining']

X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.3, random_state=42)

model2 = LogisticRegression(max_iter=1000)
model2.fit(X_train2, y_train2)

honest_score_v2 = roc_auc_score(y_test2, model2.predict_proba(X_test2)[:, 1])
print("Honest AUC score (v2, without total_clicks):", honest_score_v2)

Honest AUC score (v2, without total_clicks): 0.9636105530325828


In [28]:
honest_features_v3 = ['avg_position', 'total_sessions', 'total_engaged_sessions']

X3 = df_model[honest_features_v3]
y3 = df_model['is_declining']

X_train3, X_test3, y_train3, y_test3 = train_test_split(X3, y3, test_size=0.3, random_state=42)

model3 = LogisticRegression(max_iter=1000)
model3.fit(X_train3, y_train3)

honest_score_v3 = roc_auc_score(y_test3, model3.predict_proba(X_test3)[:, 1])
print("Honest AUC score (v3, truly independent features):", honest_score_v3)

Honest AUC score (v3, truly independent features): 0.6338426063605896


In [29]:
# THE TRAP: add a "cheat" column derived directly from the label itself
df_model['cheat_column'] = df_model['is_declining'] * 0.9 + np.random.normal(0, 0.05, len(df_model))

cheat_features = ['avg_position', 'total_sessions', 'total_engaged_sessions', 'cheat_column']

X_cheat = df_model[cheat_features]
y_cheat = df_model['is_declining']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cheat, y_cheat, test_size=0.3, random_state=42)

model_cheat = LogisticRegression(max_iter=1000)
model_cheat.fit(X_train_c, y_train_c)

cheat_score = roc_auc_score(y_test_c, model_cheat.predict_proba(X_test_c)[:, 1])
print("HONEST score (no cheat):", honest_score_v3)
print("CHEAT score (with leaked column):", cheat_score)

HONEST score (no cheat): 0.6338426063605896
CHEAT score (with leaked column): 1.0


In [30]:
# Remove the cheat column — we keep only the honest, leakage-free result
final_features = ['avg_position', 'total_sessions', 'total_engaged_sessions']
final_score = honest_score_v3

print("Final honest AUC score (leakage removed):", final_score)
print("Features used:", final_features)


Final honest AUC score (leakage removed): 0.6338426063605896
Features used: ['avg_position', 'total_sessions', 'total_engaged_sessions']


One big limitation of this work: I'm only looking at March 2026, and GA4 data is really sparse
in this slice — only about 4% of rows actually have GA4 tracking (I confirmed this in my
availability check above). Because of this, features like total_sessions and
total_engaged_sessions end up NaN for most pages, which I filled with 0 for now. That's not
ideal — a page with "0 sessions" because it genuinely had no engagement looks the same as a
page with "0 sessions" just because GA4 wasn't tracking it at all, and those are very different
situations. If I did this again, I'd either pick a month/client subset with better GA4 coverage,
or build separate models for GSC-only pages vs. pages that have both GSC and GA4 data, instead
of mixing them together.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.